In [2]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel
import re

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END

load_dotenv()

C:\Users\pakra\AppData\Local\Temp\ipykernel_22736\1555260192.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\pakra\anaconda3\envs\agentic_langgraph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
docs = (
    PyPDFLoader("./documents/book1.pdf").load()
    + PyPDFLoader("./documents/book2.pdf").load()
    + PyPDFLoader("./documents/book3.pdf").load()
)

chunks = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150
).split_documents(docs)

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
evaluator_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

generator_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
class State(TypedDict):
    question: str

    docs: List[Document]

    good_docs: List[Document]

    verdict: str
    reason: str

    strips: List[str]
    kept_strips: List[str]

    refined_context: str

    web_docs: List[Document]

    answer: str

In [ ]:
class DocEvalScore(BaseModel):
    score: float
    reason: str
doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a strict retrieval evaluator.

            Score chunk relevance from 0.0 to 1.0.

            1.0 = sufficient to answer.
            0.0 = irrelevant.

            Return score and reason.
            """
        ),
        (
            "human",
            "Question: {question}\n\nChunk:\n{chunk}"
        )
    ]
)

doc_eval_chain = (
    doc_eval_prompt
    | evaluator_llm.with_structured_output(DocEvalScore)
)

In [ ]:
UPPER_TH = 0.7
LOWER_TH = 0.3

def eval_each_doc_node(state: State):

    scores = []
    good_docs = []

    for d in state["docs"]:

        out = doc_eval_chain.invoke(
            {
                "question": state["question"],
                "chunk": d.page_content
            }
        )

        scores.append(out.score)

        if out.score > LOWER_TH:
            good_docs.append(d)

    avg_score = sum(scores) / len(scores)

    if avg_score >= UPPER_TH:
        verdict = "CORRECT"

    elif avg_score <= LOWER_TH:
        verdict = "INCORRECT"

    else:
        verdict = "AMBIGUOUS"

    return {
        "good_docs": good_docs,
        "verdict": verdict,
        "reason": f"Average score={avg_score:.2f}"
    }

In [ ]:
class KeepOrDrop(BaseModel):
    keep: Literal["yes", "no"]

In [ ]:
filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Decide whether the sentence helps answer the question.

            Return:
            keep="yes"
            or
            keep="no"
            """
        ),
        (
            "human",
            "Question: {question}\n\nSentence:\n{sentence}"
        )
    ]
)

filter_chain = (
    filter_prompt
    | evaluator_llm.with_structured_output(KeepOrDrop)
)

In [ ]:
def decompose_to_sentences(text):

    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    return [
        s.strip()
        for s in sentences
        if len(s.strip()) > 20
    ]

In [ ]:
def refine(state: State):

    if state["verdict"] == "CORRECT":

        context = "\n\n".join(
            d.page_content
            for d in state["good_docs"]
        )

    else:

        context = "\n\n".join(
            d.page_content
            for d in state["web_docs"]
        )

    strips = decompose_to_sentences(context)

    kept = []

    for s in strips:

        result = filter_chain.invoke(
            {
                "question": state["question"],
                "sentence": s
            }
        )

        if result.keep == "yes":
            kept.append(s)

    refined_context = "\n".join(kept)

    return {
        "strips": strips,
        "kept_strips": kept,
        "refined_context": refined_context,
    }

In [ ]:
tavily = TavilySearch(max_results=5)

In [ ]:
def web_search_node(state: State):

    results = tavily.invoke(
        {"query": state["question"]}
    )

    docs = []

    for r in results:

        text = f"""
TITLE: {r.get('title','')}

URL: {r.get('url','')}

CONTENT:
{r.get('content','')}
"""

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "url": r.get("url", "")
                }
            )
        )

    return {"web_docs": docs}

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Answer only from the supplied context.

            If context is insufficient,
            say "I don't know."
            """
        ),
        (
            "human",
            "Question: {question}\n\nContext:\n{refined_context}"
        )
    ]
)

In [ ]:
def generate(state: State):

    out = (
        answer_prompt
        | generator_llm
    ).invoke(
        {
            "question": state["question"],
            "refined_context": state["refined_context"]
        }
    )

    return {
        "answer": out.content
    }

In [ ]:
def route_after_eval(state: State):

    if state["verdict"] == "CORRECT":
        return "refine"

    return "web_search"

In [ ]:
g = StateGraph(State)

g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)
g.add_node("web_search", web_search_node)
g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {
        "refine": "refine",
        "web_search": "web_search",
    }
)

g.add_edge("web_search", "refine")
g.add_edge("refine", "generate")
g.add_edge("generate", END)

app = g.compile()